# Teste do Modelo 2: Salesforce Moirai

In [1]:
import pandas as pd
import torch
import os

# --- 1. Configurações ---
DATA_DIR = "../../data"
HORIZONTE_PREVISAO = 14
N_DIAS_HISTORICO = 100 # Necessário para o `context_length` do Moirai
FREQ = "D"

# --- 2. Carregar Dados ---
print("Carregando dados...")
hist_path = os.path.join(DATA_DIR, "hist.parquet")
future_cov_path = os.path.join(DATA_DIR, "future_covariates.parquet")
df_hist = pd.read_parquet(hist_path)
df_future_covariates = pd.read_parquet(future_cov_path)
print("Dados históricos e de covariáveis futuras carregados.")

# --- 3. Definir dispositivo ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsando dispositivo: {device}")

Carregando dados...
Dados históricos e de covariáveis futuras carregados.

Usando dispositivo: cpu


In [2]:
# ==============================================================================
# TESTE 2: Salesforce Moirai
# ==============================================================================
# Covariáveis: Sim. Moirai tem suporte excelente para covariáveis.
# ==============================================================================

try:
    from uni2ts.model.moirai import MoiraiForecast
    from uni2ts.common.time import PdTimestamp

    print("\n--- Testando Salesforce Moirai ---")

    # 1. Carregar o modelo
    model = MoiraiForecast.from_pretrained(
        "Salesforce/moirai-2.0-R-small",
        prediction_length=HORIZONTE_PREVISAO,
        context_length=N_DIAS_HISTORICO,
        patch_size='auto',
        device_map=device,
    )

    # 2. Preparar dados
    # Concatenar histórico + futuro para as covariáveis
    all_covariates = pd.concat([df_hist, df_future_covariates])

    data_entry = {
        "target": df_hist['target'].values,
        "start": PdTimestamp(df_hist.index[0]),
        "freq": FREQ,
        "item_id": "serie_teste_1",
        # Covariáveis conhecidas no tempo (histórico + futuro)
        "time_varying_known": all_covariates[['day_of_week', 'month']].values
    }

    # 3. Rodar a previsão
    print(f"Rodando previsão para {HORIZONTE_PREVISAO} passos...")
    forecast_df = model.predict([data_entry])

    print("Previsão (DataFrame):")
    print(forecast_df.head())
    print("Teste do Moirai concluído.\n")

except ImportError:
    print("Moirai (uni2ts) não instalado. Pulando teste.")
except Exception as e:
    print(f"Erro ao rodar Moirai: {e}")

Moirai (uni2ts) não instalado. Pulando teste.
